In [1]:
import pandas as pd
import numpy as np 


In [2]:
df = pd.read_csv('Dirty_Wine_Classification.csv')

In [4]:
print(df.shape)

(188, 17)


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 188 entries, 0 to 187
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       176 non-null    str    
 1   malic_acid                    167 non-null    float64
 2   ash                           188 non-null    float64
 3   alcalinity_of_ash             188 non-null    float64
 4   magnesium                     176 non-null    float64
 5   total_phenols                 188 non-null    float64
 6   flavanoids                    188 non-null    float64
 7   nonflavanoid_phenols          188 non-null    float64
 8   proanthocyanins               188 non-null    float64
 9   color_intensity               182 non-null    float64
 10  hue                           174 non-null    float64
 11  od280/od315_of_diluted_wines  188 non-null    float64
 12  proline                       180 non-null    float64
 13  wine_class      

In [8]:
df.describe()

,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,wine_class,optional_lab_index
count,167.000000,188.000000,188.000000,176.000000,188.000000,188.000000,188.000000,188.000000,182.000000,174.000000,188.000000,180.000000,188.000000,16.000000
mean,2.391796,2.358670,19.473404,97.107955,2.282500,2.004202,0.363404,1.582606,5.065549,0.955552,2.602713,787.538889,0.957447,4.321601
std,1.170756,0.272655,3.279940,19.627233,0.627587,1.000405,0.125473,0.561989,2.327155,0.232775,0.715194,469.609101,0.772729,3.130338
min,0.740000,1.360000,10.600000,-5.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000,0.000000,0.062294
25%,1.580000,2.207500,17.200000,88.000000,1.700000,1.097500,0.270000,1.247500,3.180000,0.762500,1.905000,508.000000,0.000000,1.933120
50%,1.880000,2.360000,19.500000,96.000000,2.310000,2.100000,0.340000,1.545000,4.750000,0.960000,2.775000,666.000000,1.000000,4.013694
75%,3.205000,2.540000,21.500000,106.250000,2.800000,2.865000,0.442500,1.927500,6.200000,1.120000,3.170000,986.250000,2.000000,6.706129
max,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,3500.000000,2.000000,8.884529


In [7]:
df.isna().sum()

alcohol                          12
malic_acid                       21
ash                               0
alcalinity_of_ash                 0
magnesium                        12
total_phenols                     0
flavanoids                        0
nonflavanoid_phenols              0
proanthocyanins                   0
color_intensity                   6
hue                              14
od280/od315_of_diluted_wines      0
proline                           8
wine_class                        0
lab_batch                         0
laboratory_comment              179
optional_lab_index              172
dtype: int64

In [12]:
df.malic_acid = df.malic_acid.fillna(df.malic_acid.median())
df.magnesium = df.magnesium.fillna(df.magnesium.median())
df.hue = df.hue.fillna(df.hue.median())
df.proline = df.hue.fillna(df.proline.median())
df.optional_lab_index = df.optional_lab_index.fillna(df.proline.median())


In [14]:
df.isna().sum()

alcohol                          12
malic_acid                        0
ash                               0
alcalinity_of_ash                 0
magnesium                         0
total_phenols                     0
flavanoids                        0
nonflavanoid_phenols              0
proanthocyanins                   0
color_intensity                   6
hue                               0
od280/od315_of_diluted_wines      0
proline                           0
wine_class                        0
lab_batch                         0
laboratory_comment              179
optional_lab_index                0
dtype: int64

In [15]:
df.alcohol =  df.alcohol.fillna(df.alcohol.mode()[0])

In [16]:
df = df.drop(columns=['laboratory_comment', 'optional_lab_index'],errors='ignore')


In [17]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)


In [18]:
X = df.drop(columns=['wine_class', 'lab_batch'])

y = df.wine_class


In [19]:
categorical_features = ['alcohol']
numeric_features = X.select_dtypes(include=np.number).columns.tolist()


In [21]:
numeric_pipeline = Pipeline([
    ('imputer' , SimpleImputer(strategy='median')),
    ('scalar',StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [23]:
models = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2000),
        {'model__C': [0.01, 0.1, 1, 10]}
    ),

    'Decision Tree': (
        DecisionTreeClassifier(random_state=42),
        {
            'model__max_depth': [None, 3, 5, 10],
            'model__min_samples_split': [2, 5],
            'model__min_samples_leaf': [1, 2]
        }
    ),

    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {
            'model__n_estimators': [100, 200],
            'model__max_depth': [None, 5, 10],
            'model__min_samples_split': [2, 5]
        }
    )
}


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
best_models = {}

for name, (classifier, params) in models.items():

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', classifier)
    ])

    grid = GridSearchCV(
        pipeline,
        params,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    results.append({
        'Model': name,
        'CV F1': grid.best_score_,
        'Best Parameters': grid.best_params_
    })

# 8. Compare models
results_df = pd.DataFrame(results)
print(results_df)

# 9. Select best model
best_name = results_df.loc[results_df['CV F1'].idxmax(), 'Model']
best_model = best_models[best_name]

print("Best model:", best_name)

# 10. Final test evaluation
y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='macro'))
print("Recall:", recall_score(y_test, y_pred, average='macro'))
print("F1 Score:", f1_score(y_test, y_pred, average='macro'))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))


                 Model     CV F1  \
0  Logistic Regression  0.931216   
1        Decision Tree  0.909840   
2        Random Forest  0.962937   

                                     Best Parameters  
0                                    {'model__C': 1}  
1  {'model__max_depth': None, 'model__min_samples...  
2  {'model__max_depth': None, 'model__min_samples...  
Best model: Random Forest
Accuracy: 0.9736842105263158
Precision: 0.9722222222222222
Recall: 0.9777777777777779
F1 Score: 0.9740129935032483
Confusion Matrix:
[[12  0  0]
 [ 0 14  1]
 [ 0  0 11]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      0.93      0.97        15
           2       0.92      1.00      0.96        11

    accuracy                           0.97        38
   macro avg       0.97      0.98      0.97        38
weighted avg       0.98      0.97      0.97        38



: 